In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import statsmodels.api as sm

In [5]:
# List of tickers to use in the analysis
tickers = ['MMM', 'ABT', 'ANF', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD',
           'AKAM', 'AA', 'ALXN', 'ATI', 'ALL', 'MO', 'AMZN', 'AEE',
           'AEP', 'AXP', 'AIG', 'AMT', 'AMP', 'ABC', 'AMGN', 'APH', 'ADI', 'AON',
           'APA', 'AIV', 'AAPL', 'AMAT', 'ADM', 'AIZ', 'T', 'ADSK', 'ADP', 'AN',
           'AZO', 'AVB', 'AVY', 'BLL', 'BAC', 'BK', 'BAX', 'BDX', 'BBBY', 'BIG',
           'BIIB', 'BLK', 'HRB', 'BA', 'BWA', 'BXP', 'BSX', 'BMY', 'CHRW', 'COG',
           'CPB', 'COF', 'CAH', 'KMX', 'CCL', 'CAT', 'CNP', 'CERN', 'CF', 'SCHW',
           'CVX', 'CMG', 'CB', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CTXS', 'CLF',
           'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'CL', 'CMCSA', 'CMA', 'CAG', 'COP',
           'CNX', 'ED', 'STZ', 'GLW', 'COST', 'CCI', 'CSX', 'CMI', 'CVS', 'DHI',
           'DHR', 'DRI', 'DVA', 'DE', 'XRAY', 'DVN', 'DFS', 'DISCA',
           'DLTR', 'D', 'RRD', 'DOV', 'DTE', 'DD', 'DUK', 'ETFC', 'EMN',
           'ETN', 'EBAY', 'ECL', 'EIX', 'EW', 'EA', 'EMR', 'ETR', 'EOG', 'EQT',
           'EFX', 'EQR', 'EL', 'EXC', 'EXPE', 'EXPD', 'XOM', 'FFIV', 'FAST', 'FDX',
           'FIS', 'FITB', 'FHN', 'FSLR', 'FE', 'FISV', 'FLIR', 'FLS', 'FLR', 'FMC',
           'FTI', 'F', 'FOSL', 'BEN', 'FCX', 'GME', 'GPS', 'GD', 'GE', 'GIS',
           'GPC', 'GNW', 'GILD', 'GS', 'GT', 'GOOG', 'GWW', 'HAL', 'HOG', 'HIG',
           'HAS', 'HP', 'HES', 'HPQ', 'HD', 'HON', 'HRL', 'HST', 'HUM', 'HBAN',
           'ITW']

# Loading data
data =  yf.download(tickers, start='2024-01-01', end='2025-01-06', auto_adjust=True)['Close']

[*********************100%***********************]  176 of 176 completed

14 Failed downloads:
['GPS', 'FLIR', 'ALXN', 'BBBY', 'COG', 'CTXS', 'ETFC', 'FISV', 'DISCA', 'CERN', 'BLL', 'RRD', 'ABC', 'BIG']: YFTzMissingError('possibly delisted; no timezone found')


In [69]:
# Remove columns (stocks) with more than 10% missing values
data = data.dropna(axis=1, thresh=len(data) * 0.9)

# As our method takes in returns, we'll calculate them from our dataframe of prices
returns = data.pct_change().dropna()

# Looking at the obtrined returns series
returns.head()

Ticker,A,AA,AAPL,ABT,ACN,ADBE,ADI,ADM,ADP,ADSK,...,ITW,KMX,KO,MMM,MO,SCHW,STZ,T,XOM,XRAY
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-03,-0.054703,-0.054638,-0.007488,-0.003004,-0.025943,-0.014274,-0.023866,0.005085,-0.003899,-0.029600,...,-0.008788,-0.048163,0.002340,-0.020091,-0.004329,-0.029667,-0.010128,-0.001159,0.008402,-0.022734
2024-01-04,-0.001220,-0.011432,-0.012700,0.013331,-0.002456,-0.008290,-0.015294,-0.018870,0.004946,0.007615,...,0.004181,-0.009651,-0.003336,0.003525,0.002899,-0.002386,0.003769,-0.004643,-0.008719,0.024698
2024-01-05,-0.003359,0.032123,-0.004013,-0.001622,-0.001394,-0.004321,0.002580,-0.013101,0.006634,0.002621,...,-0.005475,0.016288,-0.001506,0.003883,-0.000723,0.005083,0.021458,0.018659,0.003030,-0.008408
2024-01-08,0.021599,-0.012449,0.024175,0.014440,0.011081,0.028250,0.013026,0.003530,0.006590,0.025836,...,-0.002868,0.000137,0.007374,0.002486,0.010123,0.008478,0.009979,-0.008586,-0.016662,0.014698
2024-01-09,-0.020243,0.006618,-0.002263,0.002936,0.007052,0.009732,0.004127,-0.005488,-0.006504,0.002166,...,-0.003149,-0.006574,-0.001830,0.002205,-0.001193,-0.015487,0.014160,-0.005398,-0.012386,0.017549


In [70]:
t = 252

In [71]:
# Starting with setting a PCAStrategy class with 15 principal components
pca = PCA(n_components=None)

In [72]:
# To get the factor weights from PCA we'll be using a window with 252 observations as in the original paper
data_252days = returns.iloc[0:252]

# Standardize training data
standardized_252days = (data_252days - data_252days.mean()) / data_252days.std()

pca.fit(standardized_252days)

PCA()

In [73]:
K = 15

In [74]:
# Getting a 60-day window of observations to calculate factor returns
data_60days = data_252days[-60:]

# Last day in our window
data_60days.index[-1]

Timestamp('2025-01-02 00:00:00')

In [75]:
factorweights = pd.DataFrame(pca.components_, columns=data_252days.columns) / data_252days.std()

factorweights

Ticker,A,AA,AAPL,ABT,ACN,ADBE,ADI,ADM,ADP,ADSK,...,ITW,KMX,KO,MMM,MO,SCHW,STZ,T,XOM,XRAY
0,4.778612,2.893260,3.048542,4.070107,3.245526,1.454536,4.523499,2.430189,8.735679,5.108113,...,12.834410,5.696597,5.687527,3.183957,4.360251,4.654294,4.441373,3.518863,5.981487,2.861878
1,-0.296894,-1.174652,-3.294163,8.952276,-0.863020,-2.618568,-4.792530,1.002257,0.669548,-4.805977,...,-0.383208,-0.367435,20.404513,0.451578,10.556760,-1.495467,6.516918,10.049069,0.350219,0.978513
2,-7.719590,0.233892,-8.287288,-5.748623,-4.835797,-2.796083,-5.712744,2.813366,1.151089,-7.162862,...,1.495248,-1.998451,-3.951224,-0.559551,3.903983,1.904098,-5.269498,5.086904,17.919709,1.196668
3,4.543121,4.648107,2.840530,-5.333855,-5.140376,-2.373069,3.339097,2.274844,-16.237904,-3.200344,...,-4.058485,-0.343372,1.409164,-2.605586,-4.102682,-3.115567,5.555557,-1.839642,12.483772,0.213385
4,6.059810,-3.201805,2.293339,7.566593,6.491372,3.558873,2.202301,2.308702,14.158111,2.674214,...,10.078782,-0.644954,18.089025,2.920830,7.963308,-9.013205,9.662170,-0.385011,8.242792,-2.192066
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,1.298974,2.080573,-2.204406,-5.206679,-4.234820,2.595373,4.119500,-2.486739,-8.474295,-2.580662,...,11.006811,1.634058,0.429080,1.354252,-2.017877,1.953654,-7.254780,-2.042897,-3.210293,-0.963096
158,-4.580991,0.453784,1.770087,1.371408,1.346892,-3.161864,-0.401617,-4.036144,-11.181820,-0.370709,...,-12.075793,2.445771,2.852996,1.389233,6.074001,5.106777,6.115503,0.871044,-6.881479,6.674708
159,3.071257,3.666678,-0.474492,-0.543743,-3.969916,1.630462,-3.732788,1.334071,5.973742,-0.368528,...,-3.126933,0.095580,-0.757832,-3.191977,10.179257,2.175241,-1.952191,0.514042,-3.438268,1.792575
160,-0.869365,2.426310,2.359403,2.264822,1.797266,-0.078507,1.184366,1.423449,-3.758855,-0.508782,...,-5.012100,1.375300,3.106461,0.354472,-0.278827,5.622690,1.207793,-7.054294,-1.470755,-0.789540


In [76]:
# Calculating factor returns from our returns - multiplying them by factor weights
factorret = pd.DataFrame(np.dot(data_60days, factorweights.transpose()), index=data_60days.index)

# Looking at the obtained factor returns
factorret.head()

,0,1,2,3,4,5,6,7,8,9,...,152,153,154,155,156,157,158,159,160,161
Date,,,,,,,,,,,,,,,,,,,,,
2024-10-08,0.465560,1.285380,-5.505394,-6.065751,0.559987,2.029821,0.914789,1.536565,0.837911,-0.072734,...,-0.045621,-0.235434,-0.311280,-0.050591,-0.267620,0.026116,-0.067320,-0.072816,0.040788,0.032844
2024-10-09,5.404814,-1.750413,-0.303463,-2.579790,0.999536,0.689269,1.644286,0.761756,-1.799761,-0.176825,...,0.080946,-0.200187,-0.165204,0.100683,-0.141951,0.158972,-0.001519,0.090002,0.075787,-0.137982
2024-10-10,-1.679657,-2.396146,3.210841,1.289191,-0.703014,-0.724297,0.213036,1.697767,0.213221,0.039810,...,-0.088335,-0.129199,0.142893,0.174563,0.084983,0.039479,-0.205711,-0.194959,-0.031823,0.199474
2024-10-11,9.756284,-0.837857,0.246296,-2.527508,-1.520157,0.722307,-0.994341,-2.426961,0.590540,-1.198405,...,0.277871,-0.032939,0.010350,-0.225880,-0.005421,-0.262737,-0.008460,0.003181,0.132548,-0.051318
2024-10-14,4.395244,2.624517,-2.747882,-2.162697,0.728708,0.861437,-0.230790,-0.123743,-0.096396,-0.644302,...,0.220868,0.260598,-0.150656,0.067188,-0.275264,-0.127331,0.023661,0.040881,0.153213,-0.200828


In [79]:
# Dictionary to store regression results
regression_results = {}
residuals = pd.DataFrame(index=data_60days.index, columns=data_60days.columns)

# Fit regression for each stock
for stock in data_60days.columns:
    # Get stock returns
    y = data_60days[stock]
    
    # Add constant to factor returns for intercept
    X = factorret.iloc[:, :K]
    X = sm.add_constant(X)
    
    # Fit regression model
    model = sm.OLS(y, X).fit()
    
    # Store results
    regression_results[stock] = {
        'coefficients': model.params,
        'residuals': model.resid,
        'r_squared': model.rsquared,
        'p_values': model.pvalues,
        'model': model
    }
    residuals[stock] = model.resid

# Create a DataFrame with all residuals
#all_residuals = pd.DataFrame({stock: regression_results[stock]['residuals'] for stock in data_60days.columns})

# Create a DataFrame with all coefficients
#all_coefficients = pd.DataFrame({stock: regression_results[stock]['coefficients'] for stock in data_60days.columns})

residuals.head()

Ticker,A,AA,AAPL,ABT,ACN,ADBE,ADI,ADM,ADP,ADSK,...,ITW,KMX,KO,MMM,MO,SCHW,STZ,T,XOM,XRAY
Date,,,,,,,,,,,,,,,,,,,,,
2024-10-08,-0.001850,-0.007673,0.003127,0.002301,0.005066,0.004627,-0.004801,-0.002501,-0.000381,0.004173,...,-0.000242,-0.005325,-0.000084,0.004291,-0.002867,-0.001131,0.000731,0.007715,-0.005521,-0.004790
2024-10-09,-0.001741,0.002732,0.007953,-0.000320,0.005894,-0.006753,-0.000275,-0.000724,-0.001466,-0.015886,...,0.002995,-0.019271,0.001803,-0.002575,0.002728,0.011955,-0.002360,-0.002190,-0.002278,-0.006166
2024-10-10,-0.008026,0.018694,0.003024,-0.002413,0.006898,0.016904,-0.009182,-0.001850,0.002170,0.007004,...,-0.002944,-0.003346,0.002498,-0.004495,-0.006302,-0.007191,-0.004343,-0.016510,0.004013,-0.009686
2024-10-11,0.006095,0.021550,-0.010644,0.009098,-0.006256,-0.011717,-0.001742,0.007571,-0.005016,-0.006102,...,0.002265,-0.006234,0.002795,-0.003569,0.003756,0.007030,0.008552,-0.003694,-0.000127,0.007309
2024-10-14,-0.004386,0.007757,0.002362,0.001388,0.005865,0.029148,0.002750,0.002132,0.003843,0.002958,...,0.000375,0.001766,0.002022,-0.004548,-0.005468,-0.000056,0.017661,-0.016497,0.007856,-0.006970


In [ ]:
# Initialize results dictionary
asset_params = {}

k_threshold = 8.4

# Process each asset
for asset in residuals.columns:
    # Fit discrete Ornstein-Uhlenbeck process to residual time series
    """Fit discrete Ornstein-Uhlenbeck process to a time series"""
    y = residuals[asset].values[1:]
    x = residuals[asset].values[:-1]
    n = len(y)
    
    valid = False

    # Add constant to x for intercept
    X = sm.add_constant(x)
    
    # Fit OLS model
    model = sm.OLS(y, X).fit()
    
    # Extract parameters
    intercept, slope = model.params  # a is intercept, b is slope

    # Check validity
    if 0 < slope < 1:# and model.pvalues[1] < 0.05:
        valid = True
    
    if valid:
        # Calculate OU parameters
        k = -np.log(slope) * 252 # Mean reversion speed
        m = intercept / (1 - slope)  # Long-term mean
        
        # Calculate standard error
        sigma_eq = np.sqrt(np.sum(model.resid**2) / (n - 1) / (1 - slope**2))
    else:
        k, m, sigma_eq = None, None, None
    
    # Store parameters
    asset_params[asset] = {'k': k, 'm': m, 'sigma_eq': sigma_eq}

tradable_stocks = []

for asset in residuals.columns:
    # Check if mean reversion speed is sufficient
    if asset_params[asset]['k'] is not None and asset_params[asset]['k'] > k_threshold:
        tradable_stocks.append(asset)

In [83]:
s_scores = pd.DataFrame(index=residuals.index)

# Calculate S-score
for stock in tradable_stocks:
    m = asset_params[stock]['m']
    sigma_eq = asset_params[stock]['sigma_eq']
    s_scores[stock] = (residuals[stock] - m) / sigma_eq

In [91]:
last_day_scores = s_scores.iloc[-1:]

In [100]:
signals = pd.DataFrame(0, index=last_day_scores.index, columns=last_day_scores.columns)

# Long signal when S-score is below negative threshold
signals[s_scores < -1.25] = 1

# Short signal when S-score is above positive threshold
signals[s_scores > 1.25] = -1

In [108]:
# Sum loadings from components after K
portfolio_weights = np.zeros(len(returns.columns))

for i in range(K, pca.n_components_):
    portfolio_weights += pca.components_[i]

# Normalize weights to sum to 1
portfolio_weights_norm = portfolio_weights / np.sum(np.abs(portfolio_weights))

portfolio_weights_norm = pd.Series(portfolio_weights_norm, index=returns.columns)

In [111]:
current_date = returns.index[252]

In [112]:
portfolio_positions = pd.DataFrame(index=returns.index[252:], columns=returns.columns)

for stock in tradable_stocks:
    if stock in signals.columns:
        signal = signals.iloc[0][stock]  # Get signal for this stock
        if signal != 0:
            portfolio_positions.loc[current_date, stock] = signal * portfolio_weights_norm[stock]

In [119]:
portfolio_positions

Ticker,A,AA,AAPL,ABT,ACN,ADBE,ADI,ADM,ADP,ADSK,...,ITW,KMX,KO,MMM,MO,SCHW,STZ,T,XOM,XRAY
Date,,,,,,,,,,,,,,,,,,,,,
2025-01-03,NaN,NaN,-0.005965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.011209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [122]:
strategy_returns = pd.Series(index=returns.index[252:], dtype=float)
if t+1 <= len(returns):
    next_day_returns = returns.iloc[t]
    day_return = 0
    
    for stock in tradable_stocks:
        position = portfolio_positions.loc[current_date, stock]
        if not np.isnan(position) and position != 0:
            day_return += position * next_day_returns[stock]
    
    strategy_returns.loc[current_date] = day_return

In [123]:
strategy_returns

Date
2025-01-03   -0.000289
dtype: float64